# 🌿 Ain – AI-Powered Municipal Infrastructure Assistant

## 📌 Project Overview

Ain is an AI-powered municipal infrastructure assistant designed to automate the processing of citizen reports. Instead of manually reviewing unstructured text, the system analyzes reports, extracts key information, validates the results, and stores them in a structured format for easier management.

The project is built using **CrewAI** with a multi-agent architecture. The first agent analyzes the citizen's report and extracts important details such as the infrastructure issue, location, priority level, and description. The second agent reviews the extracted information to ensure accuracy, consistency, and proper formatting before generating the final output.

The application is connected to **OpenRouter** to access a Large Language Model (LLM), **Gradio** to provide an interactive user interface, and **Google Sheets API** to automatically store validated reports in a structured database.

---

## 🎯 Project Objectives

- Automate the analysis of municipal infrastructure reports.
- Convert unstructured text into structured data.
- Improve report accuracy through multi-agent validation.
- Store processed reports automatically in Google Sheets.
- Provide a simple and user-friendly interface for submitting reports.

---

## 🛠 Technologies Used

- Python
- CrewAI
- OpenRouter
- Gradio
- Google Sheets API

---

## 🤖 AI Agents

### Report Analyzer
Responsible for analyzing the citizen's report and extracting:
- Infrastructure issue type
- Location
- Priority level
- Description

### Quality Reviewer
Responsible for reviewing the extracted information, validating its accuracy and completeness, and ensuring the output follows the required JSON structure before saving.

---

## 🔄 Workflow

1. The user submits a municipal report through the Gradio interface.
2. CrewAI starts the multi-agent workflow.
3. The Report Analyzer processes the report and extracts structured information.
4. The Quality Reviewer validates and reviews the extracted data.
5. The validated report is automatically saved to Google Sheets.
6. The final structured result is displayed to the user.

---

## ✅ Expected Outcome

Ain reduces manual effort in processing municipal reports, improves data consistency, and demonstrates how AI agents can support smarter municipal infrastructure management through automated analysis, validation, and structured data storage.

# Part 1: Environment Setup

## Step 1: Install CrewAI

In [27]:
!pip install -q -U crewai

## Step 2: Import the Required Classes

In [28]:
import os

from crewai import Agent, Task, Crew, Process, LLM

## Step 3: Configure How to Run the LLM

### OpenRouter

Use [OpenRouter](https://openrouter.ai) to access many models (OpenAI, Anthropic, Meta, etc.) through a single API key.

**Best for:** Anyone who wants one API key with access to multiple model providers.

**Before running:** Replace `YOUR_OPENROUTER_API_KEY` with your key from [openrouter.ai/keys](https://openrouter.ai/keys).

In [29]:
os.environ["OPENROUTER_API_KEY"] = "YOUR_OPENROUTER_API_KEY"  # <-- paste your own key from openrouter.ai/keys, never commit a real key

llm = LLM(
    model="openrouter/openai/gpt-4o-mini",
    base_url="https://openrouter.ai/api/v1",
    temperature=0
)

# Part 2: Prepare the Citizen Report

## Step 4: Define the Citizen Report

In [30]:
citizen_report = (
    "There is a large pothole near a school in Al Narjis District, Riyadh."
)

## Step 5 (optional): Example of a Second Test Report

You can swap in any citizen report text below to test the pipeline with different cases.

In [31]:
# Example of an alternate report you can try instead of the one above
alt_citizen_report = (
    "There has been water leaking from a broken pipe on King Fahd Road for two days."
)

# Part 3: Build the Crew

## Step 7: Create the Report Analyzer Agent

In [32]:
report_analyzer = Agent(
    role="Infrastructure Report Analyzer",

    goal=(
        "Analyze citizen infrastructure reports and extract the issue type, "
        "location, description, and responsible department."
    ),

    backstory=(
        "You are a careful municipal report analyst. "
        "You organize citizen reports into clear structured information. "
        "You do not invent missing details."
    ),

    llm=llm,
    verbose=True,
    allow_delegation=False
)

## Step 8: Create the Quality Reviewer Agent

In [33]:
quality_reviewer = Agent(
    role="Infrastructure Quality Reviewer",
    goal=(
        "Review the analyzed citizen infrastructure report, verify that all "
        "required information is complete, assign a priority level, and "
        "prepare the final validated report."
    ),
    backstory=(
        "You are an AI reviewer for municipal infrastructure reports. "
        "You receive the analysis from another agent and check that it is "
        "complete and consistent with the original citizen report. "
        "You assign a priority level (High, Medium, or Low), provide a "
        "confidence score, and prepare the final report before it is saved."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False
)

## Agent Comparison

| Agent | Main Responsibility |
|---|---|
| Report Analyzer | Extract issue type, location, and description from the citizen report |
| Quality Reviewer | Validate the extracted data, assign priority, and prepare the final report |

Both agents use the same model, but their different roles and goals guide their behavior.

## Step 9: Create the Research Task

In [34]:
research_task = Task(
    description="""
Analyze the following citizen infrastructure report:

{citizen_report}

Your job is to analyze the report and extract only the information explicitly mentioned.

Prepare the analysis using the following sections:

1. Issue Type
2. Location (if mentioned)
3. Short Description

Rules:

- Extract only the information available in the report.
- Do not invent or assume missing details.
- If the location is not mentioned, write "Not Provided".
- Keep the description short and clear.
- Do NOT assign a priority level.
- Do NOT assign the responsible department.
""",

    expected_output=(
        "A structured report containing:\n"
        "- Issue Type\n"
        "- Location\n"
        "- Short Description"
    ),

    agent=report_analyzer
)

## Step 10: Create the Review Task

The Quality Reviewer receives the Report Analyzer's draft through:

```python
context=[research_task]
```

In [35]:
review_task = Task(
    description="""
Review the analyzed citizen infrastructure report.

Use the output from Agent 1 and perform the following:

1. Verify that all required information is present.
2. Check that the issue type matches the original report.
3. Verify that the location is correct or marked as "Not Provided".
4. Ensure the short description is clear and complete.
5. Assign a priority level:
   - High
   - Medium
   - Low
6. Explain the reason for the assigned priority.
7. Provide a confidence score (0–100).
8. Prepare the final validated infrastructure report.

Rules:

- Do not invent missing information.
- Base your review only on the original citizen report.
- Keep the final report clear and concise.
- Return ONLY valid JSON.
- Do not include markdown, explanations, or extra text.
""",

    expected_output="""
{
  "issue_type": "string",
  "location": "string",
  "description": "string",
  "priority": "High | Medium | Low",
  "priority_reason": "string",
  "confidence_score": 95,
  "status": "Ready for Submission"
}
""",

    agent=quality_reviewer,
    context=[research_task]
)

## Step 11: Assemble the Sequential Crew

In [36]:
ain_crew = Crew(
    agents=[
        report_analyzer,
        quality_reviewer
    ],

    tasks=[
        research_task,
        review_task
    ],

    process=Process.sequential,
    verbose=True
)

print("Ain multi-agent crew created successfully.")

Ain multi-agent crew created successfully.


# Part 4: Run & Review Results

## Step 12: Start the Crew

In Jupyter, the notebook already runs an async event loop, so use `await crew.kickoff_async()` instead of `kickoff()`.

Watch the logs to observe:

1. The Researcher completing the first task
2. The handoff to the Critic
3. The Critic reviewing the draft
4. The final reviewed output

In [37]:
crew_result = await ain_crew.kickoff_async()

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 7b572e4c-6f4f-4641-a8ec-50558aaa020c                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│  Analyze the following citizen infrastructure report:                                                           │
│                                                                                                                 │
│  {citizen_report}                                                                                               │
│                                                                                                                 │
│  Your job is to analyze the report and extract only the information explicitly mentioned.                       │
│                                                                                                                 │
│  Prepare the analysis using the following sections:                                                             │
│                                                                                                                 │
│  1. Issue Type                                                                                                  │
│  2. Location (if mentioned)                                                                                     │
│  3. Short Description                                                                                           │
│                                                                                                                 │
│  Rules:                                                                                                         │
│                                                                                                                 │
│  - Extract only the information available in the report.                                                        │
│  - Do not invent or assume missing details.                                                                     │
│  - If the location is not mentioned, write "Not Provided".                                                      │
│  - Keep the description short and clear.                                                                        │
│  - Do NOT assign a priority level.                                                                              │
│  - Do NOT assign the responsible department.                                                                    │
│                                                                                                                 │
│  ID: e8888056-c261-42e4-a81c-b1df22497033                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Infrastructure Report Analyzer                                                                          │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Analyze the following citizen infrastructure report:                                                           │
│                                                                                                                 │
│  {citizen_report}                                                                                               │
│                                                                                                                 │
│  Your job is to analyze the report and extract only the information explicitly mentioned.                       │
│                                                                                                                 │
│  Prepare the analysis using the following sections:                                                             │
│                                                                                                                 │
│  1. Issue Type                                                                                                  │
│  2. Location (if mentioned)                                                                                     │
│  3. Short Description                                                                                           │
│                                                                                                                 │
│  Rules:                                                                                                         │
│                                                                                                                 │
│  - Extract only the information available in the report.                                                        │
│  - Do not invent or assume missing details.                                                                     │
│  - If the location is not mentioned, write "Not Provided".                                                      │
│  - Keep the description short and clear.                                                                        │
│  - Do NOT assign a priority level.                                                                              │
│  - Do NOT assign the responsible department.                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Infrastructure Report Analyzer                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {citizen_report}                                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Infrastructure Quality Reviewer                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "issue_type": "Pothole",                                                                                     │
│    "location": "Main St and 2nd Ave",                                                                           │
│    "description": "Large pothole causing damage to vehicles and posing a safety hazard.",                       │
│    "priority": "High",                                                                                          │
│    "priority_reason": "The pothole is large and poses a significant safety risk to vehicles and pedestrians,    │
│  requiring immediate attention.",                                                                               │
│    "confidence_score": 90,                                                                                      │
│    "status": "Ready for Submission"                                                                             │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Review the analyzed citizen infrastructure report.                                                             │
│                                                                                                                 │
│  Use the output from Agent 1 and perform the following:                                                         │
│                                                                                                                 │
│  1. Verify that all required information is present.                                                            │
│  2. Check that the issue type matches the original report.                                                      │
│  3. Verify that the location is correct or marked as "Not Provided".                                            │
│  4. Ensure the short description is clear and complete.                                                         │
│  5. Assign a priority level:                                                                                    │
│     - High                                                                                                      │
│     - Medium                                                                                                    │
│     - Low                                                                                                       │
│  6. Explain the reason for the assigned priority.                                                               │
│  7. Provide a confidence score (0–100).                                                                         │
│  8. Prepare the final validated infrastructure report.                                                          │
│                                                                                                                 │
│  Rules:                                                                                                         │
│                                                                                                                 │
│  - Do not invent missing information.                                                                           │
│  - Base your review only on the original citizen report.                                                        │
│  - Keep the final report clear and concise.                                                                     │
│  - Return ONLY valid JSON.                                                                                      │
│  - Do not include markdown, explanations, or extra text.                                                        │
│                                                                                                                 │
│  Agent: Infrastructure Quality Reviewer                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 7b572e4c-6f4f-4641-a8ec-50558aaa020c                                                                       │
│  Final Output: {                                                                                                │
│    "issue_type": "Pothole",                                                                                     │
│    "location": "Main St and 2nd Ave",                                                                           │
│    "description": "Large pothole causing damage to vehicles and posing a safety hazard.",                       │
│    "priority": "High",                                                                                          │
│    "priority_reason": "The pothole is large and poses a significant safety risk to vehicles and pedestrians,    │
│  requiring immediate attention.",                                                                               │
│    "confidence_score": 90,                                                                                      │
│    "status": "Ready for Submission"                                                                             │
│  }                                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Step 13: Display the Final Reviewed Report

In [38]:
print(crew_result.raw)

{
  "issue_type": "Pothole",
  "location": "Main St and 2nd Ave",
  "description": "Large pothole causing damage to vehicles and posing a safety hazard.",
  "priority": "High",
  "priority_reason": "The pothole is large and poses a significant safety risk to vehicles and pedestrians, requiring immediate attention.",
  "confidence_score": 90,
  "status": "Ready for Submission"
}


## Step 14: Display the Researcher's Draft

In [39]:
analyzer_output = crew_result.tasks_output[0]

print("INFRASTRUCTURE REPORT ANALYZER OUTPUT")
print("=" * 60)
print(analyzer_output.raw)

INFRASTRUCTURE REPORT ANALYZER OUTPUT
{citizen_report}


## Step 15: Display the Critic's Final Report

In [40]:
reviewer_output = crew_result.tasks_output[1]

print("INFRASTRUCTURE QUALITY REVIEW")
print("=" * 60)
print(reviewer_output.raw)

INFRASTRUCTURE QUALITY REVIEW
{
  "issue_type": "Pothole",
  "location": "Main St and 2nd Ave",
  "description": "Large pothole causing damage to vehicles and posing a safety hazard.",
  "priority": "High",
  "priority_reason": "The pothole is large and poses a significant safety risk to vehicles and pedestrians, requiring immediate attention.",
  "confidence_score": 90,
  "status": "Ready for Submission"
}


In [41]:
!pip install -q gspread google-auth

In [42]:
from google.colab import auth
auth.authenticate_user()

In [43]:
import gspread
from google.auth import default

creds, _ = default()
gc = gspread.authorize(creds)

sheet = gc.open("Ain Infrastructure Reports").sheet1

print("Google Sheet connected successfully.")

Google Sheet connected successfully.


In [44]:
def save_to_sheet(issue_type, location, description,
                  priority, priority_reason,
                  confidence_score, status):

    sheet.append_row([
        issue_type,
        location,
        description,
        priority,
        priority_reason,
        confidence_score,
        status
    ])

    print("Report saved successfully!")

In [45]:
save_to_sheet(
    issue_type="Pothole",
    location="Al Narjis District, Riyadh",
    description="Large pothole near a school",
    priority="High",
    priority_reason="The issue may affect pedestrian and student safety.",
    confidence_score=95,
    status="Ready for Submission"
)

Report saved successfully!


In [46]:
!pip install -q gradio

In [47]:
import gradio as gr

In [48]:
import json

async def process_report(citizen_report):
    result = await ain_crew.kickoff_async(
        inputs={"citizen_report": citizen_report}
    )

    data = json.loads(str(result))

    save_to_sheet(
        issue_type=data["issue_type"],
        location=data["location"],
        description=data["description"],
        priority=data["priority"],
        priority_reason=data["priority_reason"],
        confidence_score=data["confidence_score"],
        status=data["status"]
    )

    return json.dumps(data, indent=4)

In [63]:
demo.launch()

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3c689a184cd9efc7b3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [62]:
import gradio as gr

custom_css = custom_css = """
body {
    background: #F1FFF3 !important;
}

.gradio-container {
    background: #F1FFF3 !important;
}

/* العنوان */
h1 {
    color: #1B5E20 !important;
    font-weight: bold !important;
}

/* الوصف */
.prose,
.prose p {
    color: #1B5E20 !important;
}

/* أسماء الحقول */
label {
    color: #1B5E20 !important;
    font-weight: bold !important;
}

/* مربعات الإدخال والإخراج */
textarea,
input {
    background: white !important;
    color: #1B5E20 !important;
    border: 2px solid #66BB6A !important;
    border-radius: 10px !important;
}

/* زر Submit */
button.primary {
    background: #2E7D32 !important;
    color: white !important;
    border: none !important;
}

/* عند المرور بالماوس */
button.primary:hover {
    background: #1B5E20 !important;
}

/* زر Clear */
button.secondary {
    background: #2E7D32 !important;
    color: white !important;
    border: none !important;
}

footer {
    display: none !important;
}
"""

demo = gr.Interface(
    fn=process_report,
    inputs=gr.Textbox(
        lines=6,
        label="Citizen Report",
        placeholder="Describe the infrastructure issue..."
    ),
    outputs=gr.Textbox(
        lines=12,
        label="Validated Report"
    ),
    title="Ain - Municipal Infrastructure Assistant",
    description="Submit a municipal infrastructure report for AI analysis.",
    css=custom_css
)

demo.launch()

/usr/local/lib/python3.12/dist-packages/gradio/interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  super().__init__(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3c689a184cd9efc7b3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
